In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt


In [3]:
# Load past and future data
past_data = pd.read_csv('../../data/vsf_ml_models/data/indexes_past_data.csv')
future_data = pd.read_csv('../../data/vsf_ml_models/data/indexes_future_data.csv')

# Parse dates
past_data['Date'] = pd.to_datetime(past_data['Date'], format='%d-%m-%Y')
future_data['Date'] = pd.to_datetime(future_data['Date'], format='%d-%m-%Y')


In [5]:
# Create a 'Quarter' feature
past_data['Quarter'] = past_data['Date'].dt.to_period('Q')
future_data['Quarter'] = future_data['Date'].dt.to_period('Q')

# Manually assume future CS (e.g., 1.65 billion for all quarters in 2025)
assumed_future_cs = 1.65e9  # 1.65 billion
future_data['CS'] = assumed_future_cs

# Calculate derived VS for future data based on the assumed percentage (9%)
future_data['VS'] = future_data['CS'] * 0.09  # Example: 9% of CS

# Aggregate data quarterly for both datasets
past_data_quarterly = past_data.groupby('Quarter').mean().reset_index()
future_data_quarterly = future_data.groupby('Quarter').mean().reset_index()


In [6]:
# Features and target
X = past_data_quarterly[['AVPI', 'LAR', 'OP_Price', 'PR', 'PSI', 'VPI', 'CS']]
y = past_data_quarterly['VS']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:
# Initialize and train XGBoost Regressor
model = XGBRegressor(
    objective='reg:squarederror', 
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42
)
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")


RMSE: 6302665.562087908
MAE: 6302665.562087908


In [8]:
# Prepare future data for prediction
future_X = future_data_quarterly[['AVPI', 'LAR', 'OP_Price', 'PR', 'PSI', 'VPI', 'CS']]

# Predict VS for 2025
future_data_quarterly['Predicted_VS'] = model.predict(future_X)

# Display predicted results
print(future_data_quarterly[['Quarter', 'Predicted_VS']])


  Quarter  Predicted_VS
0  2024Q4    79664104.0
1  2025Q1    79664104.0
2  2025Q2    79664104.0
3  2025Q3    79664104.0
4  2025Q4    79664104.0
